In [0]:
from pyspark.sql import functions as F

MY_ID = "Musarraf_7045"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

silver_g = spark.table(f"workspace.capstone_{MY_ID}.silver_gateway")
silver_l = spark.table(f"workspace.capstone_{MY_ID}.silver_ledger")

In [0]:
g = silver_g.alias("g")
l = silver_l.alias("l")

joined = g.join(l, on="txn_id", how="full_outer")

gold = (joined
    .withColumn("gateway_date", F.col("g.settlement_date"))
    .withColumn("ledger_date", F.col("l.settlement_date"))
    .withColumn("amount_diff", F.round(F.abs(F.col("g.amount") - F.col("l.amount")), 2))
    .withColumn("date_diff_days", F.datediff(F.col("l.settlement_date"), F.col("g.settlement_date")))
    .withColumn("match_status",
        F.when(F.col("g.txn_id").isNull(), "LEDGER_ONLY")
         .when(F.col("l.txn_id").isNull(), "GATEWAY_ONLY")
         .when(F.col("g.amount").isNull(), "GATEWAY_AMOUNT_MISSING")
         .when(F.col("date_diff_days") != 0, "DATE_MISMATCH")
         .when(F.col("amount_diff") > 0.05, "AMOUNT_MISMATCH")
         .otherwise("MATCHED"))
    .withColumn("break_reason",
        F.expr("""CASE match_status
                    WHEN 'MATCHED' THEN NULL
                    ELSE match_status END"""))
    .select(
        F.coalesce(F.col("g.txn_id"), F.col("l.txn_id")).alias("txn_id"),
        "match_status", "break_reason",
        F.col("g.amount").alias("gateway_amount"),
        F.col("l.amount").alias("ledger_amount"),
        "amount_diff",
        F.col("g.txn_ts").alias("gateway_ts"),
        F.col("l.txn_ts").alias("ledger_ts"),
        "date_diff_days",
        F.coalesce("gateway_date", "ledger_date").alias("settlement_date"),
        F.coalesce(F.col("g.merchant_id"), F.col("l.merchant_id")).alias("merchant_id"),
        F.coalesce(F.col("g.payment_method"), F.col("l.payment_method")).alias("payment_method"),
    )
)

gold.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.gold_reconciliation")

In [0]:
gold.groupBy("match_status").count().orderBy(F.desc("count")).show()
print("total rows:", gold.count())

+--------------------+-----+
|        match_status|count|
+--------------------+-----+
|             MATCHED|39900|
|     AMOUNT_MISMATCH|  450|
|GATEWAY_AMOUNT_MI...|  350|
|       DATE_MISMATCH|  300|
|        GATEWAY_ONLY|  208|
|         LEDGER_ONLY|  193|
+--------------------+-----+

total rows: 41401


###EXPORT

In [0]:
for name, df in [("silver_gateway", silver_g), ("silver_ledger", silver_l)]:
    (df.coalesce(1)
       .write.mode("overwrite")
       .option("header", True)
       .csv(f"{VOL}/export/{name}"))